## Installation

In [ ]:
%pip install -r requirements.txt --upgrade

#https://huggingface.co/docs/transformers/installation#install-from-source
!apt install python3.10-venv
!python -m venv .env
!source .env/bin/activate
%pip install torch torchvision
%pip install tensorflow
%pip install tf-keras
%pip install biopython

%pip install git+https://github.com/huggingface/transformers

In [ ]:
%pip list

In [ ]:
%pip install biotite

## File Partitioning

In [ ]:
import re
import textwrap
output_file_path = "validation_M6.txt"

input_file = open('tps_larger_training_set_M6.fsa', 'r')
lines = input_file.readlines()
output_lines = []
for line in lines:
  if line.startswith(">"):
    output_lines.append("<|endoftext|>")
  else:
    chunked_string = textwrap.wrap(line, width=60)
    for chunk in chunked_string:
      output_lines.append(chunk)

with open(output_file_path, 'w') as file:
    for item in output_lines:
         file.write(item + "\n") 

In [ ]:

import re
import textwrap

files = ['tps_larger_training_set_M1.fsa',
         'tps_larger_training_set_M2.fsa',
         'tps_larger_training_set_M3.fsa',
         'tps_larger_training_set_M4.fsa',
         'tps_larger_training_set_M5.fsa'
         ]

output_lines_train = []
for i in files:
  input_file = open(i, 'r')
  lines = input_file.readlines()

  for l in lines:
    if l.startswith(">"):
      output_lines_train.append("<|endoftext|>")
    else:
      chunked_string = textwrap.wrap(l, width=60)
      for chunk in chunked_string:
        output_lines_train.append(chunk)
  
output_file_path = "training.txt"

with open(output_file_path, 'w') as file:
    for item in output_lines_train:
        file.write(item + "\n")

## Sequence file to fasta format

In [ ]:
def convert_to_fasta(input_filename, output_filename):
    """
    Converts a text file containing protein sequences, 
    one sequence per line, into a fasta format file.

    Args:
        input_filename (str): Name of the input file.
        output_filename (str): Name of the output file.
    """
    with open(input_filename, 'r') as input_file, \
         open(output_filename, 'w') as output_file:
        sequence_id = 1
        for line in input_file:
            sequence = line.strip()
            if sequence:  # Ignore empty lines
                output_file.write(f">Sequence_{sequence_id}\n{sequence}\n")
                sequence_id += 1


# Example usage:
input_filename = "tps_larger_training_set.txt"
output_filename = "tps_larger_training_set.fasta"

#convert_to_fasta(input_filename, output_filename)

## Fine Tuning ProtGPT2

In [1]:
# https://huggingface.co/nferruz/ProtGPT2
# https://huggingface.co/littleworth/protgpt2-distilled-tiny
!python run_clm.py --model_name_or_path littleworth/protgpt2-distilled-tiny \
--train_file training.txt \
--validation_file validation_M6.txt \
--tokenizer_name littleworth/protgpt2-distilled-tiny \
--do_train --do_eval \
--eval_strategy epoch \
--output_dir ./output_tiny/ \
--learning_rate 1e-4 \
--overwrite_output_dir \
--per_device_train_batch_size 64 \
--per_device_eval_batch_size 64 \
--block_size 512 \
--max_steps 4000 \
--gradient_accumulation_steps 8

2025-09-13 23:31:38.092855: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-09-13 23:31:38.615719: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1757806298.814555   14064 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1757806298.862780   14064 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1757806299.302781   14064 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

## TPS Generation

In [ ]:
from transformers import pipeline, GPT2LMHeadModel, GPT2TokenizerFast
import torch
import math

# https://huggingface.co/nferruz/ProtGPT2
# https://huggingface.co/nferruz/ProtGPT2/discussions/38
# Generate around 1000 sequences
# calculate perplexity scores
# sort them by lowest perplexity & take top 10%

TOTAL_SEQUENCES = 1000
NUM_SEQ_PER_GENERATION = 50

model_path = "./output_tiny/"
device = 'cuda'

def calculatePerplexity(sequence, model, tokenizer):
    input_ids = torch.tensor(tokenizer.encode(sequence)).unsqueeze(0) 
    input_ids = input_ids.to(device)
    with torch.no_grad():
        outputs = model(input_ids, labels=input_ids)
    loss, logits = outputs[:2]
    return math.exp(loss)

def sort_protein_sequences(sequences, scores, top_n=1000):
    """
    Sorts protein sequences by score and returns the top N sequences.

    Args:
        sequences (list): List of protein sequences.
        scores (list): List of scores corresponding to protein sequences.
        top_n (int, optional): Number of top sequences to return. Defaults to 1000.

    Returns:
        list: Top N protein sequences along with their scores.
    """
    # Combine sequences and scores into a list of tuples
    sequence_score_pairs = list(zip(sequences, scores))
    # Sort the list in ascending order based on scores
    sorted_pairs = sorted(sequence_score_pairs, key=lambda x: x[1], reverse=False)
    return sorted_pairs[:top_n]

# ProtGPT2 model
model = GPT2LMHeadModel.from_pretrained(model_path).to(device)
# ProtGPT2 tokenizer
tokenizer = GPT2TokenizerFast.from_pretrained(model_path)
# protgpt2 pipeline to generate artificial TPS sequences from the fine tuned model
protgpt2 = pipeline('text-generation', model=model_path, device=device)

 # integer division
num_iter = TOTAL_SEQUENCES // NUM_SEQ_PER_GENERATION
# generated TPS proteins
tps_proteins = []
# perplexity scores for corresponding proteins
ppl_scores = []

for i in range(0, num_iter):
  # generate sequences given the <|endoftext|> prompt
  sequences = protgpt2(
    "<|endoftext|>", 
    max_length=100, # length is expressed in tokens, where each token has an average length of 4 amino acids.
    do_sample=True, 
    top_k=950, 
    repetition_penalty=1.2, 
    num_return_sequences=NUM_SEQ_PER_GENERATION, 
    pad_token_id=tokenizer.eos_token_id,  # Set pad_token_id to eos_token_id
    eos_token_id=0, 
    truncation=True)
  # compute perplexity scores for each generated sequence and store them
  for seq in sequences:
    tmp_seq = seq['generated_text'].rstrip()
    modified_seq_padded = tmp_seq + '<|endoftext|>'
    tps_proteins.append(tmp_seq + '\n')
    ppl = calculatePerplexity(modified_seq_padded, model, tokenizer)
    print("Generated ", str(len(tps_proteins)), " sequences")
    ppl_scores.append(ppl)

# pick top 300
top_sequences = sort_protein_sequences(tps_proteins, ppl_scores, top_n=300)

# write the generated sequences to file
output_seq = []
output_ppl_scores = []
for sequence, score in top_sequences:
    output_seq.append(sequence)
    output_ppl_scores.append(round(score, 2))

output_file_path = "top_tps_sequences.txt"
with open(output_file_path, 'w') as file:
    for item in output_seq:
         file.write(item.rstrip() + '\n')

output_file_path = "perplexity_scores.txt"
with open(output_file_path, 'w') as file:
    for item in output_ppl_scores:
         file.write(str(item) + '\n')

## Helper Functions

In [ ]:
# code to extract the generated sequences into an array
def parse_sequences(filename):
    parsed_array = []
    start_value = "<|endoftext|>"
    previous_line = None
    is_seq_line = False
    current_seq = ""
    with open(filename, 'r') as file:
        for line in file:
            if line.startswith(start_value) == False:
                current_seq += line.rstrip()
            else: # True
                if current_seq != '':
                    parsed_array.append(current_seq)
                    current_seq = ''
        # append the last sequence
        parsed_array.append(current_seq)

    return parsed_array

def read_scores(filename):
    scores = []
    with open(filename, 'r') as file:
        for line in file:
            scores.append(float(line.rstrip()))

    return scores

from Bio import pairwise2
from Bio.pairwise2 import format_alignment
import random

# Sequence identity based on Appendix.A1 here: https://arxiv.org/pdf/2404.10457
from Bio import Align
from Bio.Align import substitution_matrices

aligner = Align.PairwiseAligner()
aligner.open_gap_score = -11
aligner.extend_gap_score = -1
aligner.substitution_matrix = substitution_matrices.load("BLOSUM62")

def pairwise_align_score(seq0: str, seq1: str) -> float:
    """Calculate sequence identity between two protein sequences with a BioPython Aligner."""
    alignments = aligner.align(seq0, seq1)
    alignment = next(alignments)
    
    parts = alignment.format("fasta").split('\n')
    seq0, seq1 = parts[1], parts[3]
    matches = sum(1 for a, b in zip(seq0, seq1) if a == b)
    percent_match = matches / min(len(seq0), len(seq1))
    return percent_match

print(pairwise_align_score("ACDQAA", "ACNQ"))


## Sequence Identity

In [ ]:


# read the generated tps sequences into an array
new_tps_seq = parse_sequences('top_tps_sequences.txt')
# read the training sequences into an array
training_parse_seq = parse_sequences('training.txt')
print(len(training_parse_seq))

results = []
max_identity = []
count = 0
for seq1 in new_tps_seq:
    max_var = 0
    # pick 200 random sequences from the training
    random_sequences = random.sample(training_parse_seq, 100)
    for seq2 in random_sequences:
        identity = pairwise_align_score(seq1, seq2)
        count +=1
        if identity > max_var:
            max_var = identity
        results.append(identity)
        if count % 100 == 0: # very good condition check!
            print("completed ", count, " training sequences")
    max_identity.append(max_var)

with open('max_identity_scores.txt', 'w') as file:
    for score in max_identity:
        file.write(str(score) + '\n')

## DataFrame creation

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# read all columns
new_tps_seq = parse_sequences('top_tps_sequences.txt')
ppl_scores = read_scores('perplexity_scores.txt')
#max_identity_scores = read_scores('max_identity_scores.txt')
# create dataframe
df = pd.DataFrame(list(zip(new_tps_seq, ppl_scores)), columns=['sequence', 'perplexity'])
df.to_csv('top_tps_seq_scores.csv', index=False) 
#df['identity_score'].mean()


## Write final sequences to fasta file

In [ ]:
import pandas as pd 

# read the probable TPS sequences from the csv file
df1 = pd.read_csv('top_tps_seq_scores.csv')
filtered = df1#[df1['identity_score'] <= 0.35]


In [ ]:
# write the sequences to a fasta file
idx = 1
with open("filtered_tps_seq.fasta", "w") as f:
    for index, row in filtered.iterrows():
        f.write(">" + str(idx) + '\n')
        f.write(row['sequence'] + '\n')
        idx += 1


## Miscellaneous

In [ ]:
import torch
import esm
from transformers import AutoTokenizer, EsmForProteinFolding
import biotite.structure.io as bsio
import matplotlib.pyplot as plt
import time
import argparse

torch.backends.cuda.matmul.allow_tf32 = True

def pLDDT(sequence, model):
    with torch.no_grad():
        output = model.infer_pdb(sequence)
    with open("result.pdb", "w") as f:
        f.write(output)

    struct = bsio.load_structure("result.pdb", extra_fields=["b_factor"])
    return struct.b_factor.mean()  # this will be the pLDDT

def writePDB(sequence, model, filepath):
    print(f"Writing PDB for {filepath} (length {len(sequence)})")
    with torch.no_grad():
        output = model.infer_pdb(sequence)
    with open(filepath, "w") as f:
        f.write(output)

def get_esm_model():
    model = EsmForProteinFolding.from_pretrained("facebook/esmfold_v1", low_cpu_mem_usage=True)
    model.esm = model.esm.half()
    model.trunk.set_chunk_size(8)

    device = None
    # Check if MPS is available (Metal Performance Shaders)
    if torch.backends.mps.is_available():
        print("MPS is available!")
        device = torch.device('mps') # Apple M1/M2 machines
    elif torch.cuda.is_available():
        print("CUDA is available!")
        device = torch.device('cuda')
    else:
        print("Running on CPU")

    if device is not None:
        model = model.to(device)

    model = model.eval()
    return model


In [2]:
from pathlib import Path

def parse_fasta(fasta_path):
    """
    Parses a FASTA file and yields (seq_id, sequence) tuples.
    """
    with open(fasta_path, "r") as f:
        seq_id = None
        seq_chunks = []
        for line in f:
            line = line.strip()
            if not line:
                continue
            if line.startswith(">"):
                if seq_id:
                    yield seq_id, "".join(seq_chunks)
                seq_id = line[1:].split()[0]  # take first word after '>'
                seq_chunks = []
            else:
                seq_chunks.append(line)
        if seq_id:
            yield seq_id, "".join(seq_chunks)

def process_fasta(fasta_path, model, output_dir="pdb_outputs"):
    """
    Reads FASTA, and for each sequence calls writePDB.
    """
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)

    for seq_id, sequence in parse_fasta(fasta_path):
        out_file = output_dir / f"{seq_id}.pdb"
        if out_file.exists():
            print(f"Skipping {seq_id}: {out_file} already exists")
            continue
        writePDB(sequence, model, str(out_file))

In [ ]:
fasta_file = '/teamspace/studios/this_studio/predicted_tps_sequences.fasta'
model = get_esm_model()
process_fasta(fasta_file, model, '/teamspace/studios/this_studio/query_pdbs')

In [ ]:
fasta_file = '/teamspace/studios/this_studio/training_top_seq_hits_foldseek.fasta'
model = get_esm_model()
process_fasta(fasta_file, model, '/teamspace/studios/this_studio/target_pdbs')

In [ ]:
# Code to get max seq identity with the training set and pLDDT for a given sequence.

model = get_esm_model()

# potential TPS sequences
indices = [
    56, 206, 435, 211, 143, 74, 102, 284, 172, 70,
    270, 175, 58, 158, 318, 121, 585, 1159, 252, 231,
    131, 576, 108, 530, 485, 118, 109, 272, 521, 23,
    308, 193, 38, 120, 104
]
training_parse_seq = parse_sequences('training.txt')
output = { 'maxID': [], 'index': [], 'seq': [], 'pLDDT' :[]}

for idx in indices:
    _idx = idx - 1 # dataframe indexes from 0.
    target_seq = filtered.iloc[_idx]['sequence']
    print('seq =', target_seq)
    print('idx = ', idx, '| len = ', len(target_seq))
    
    output['index'].append(idx)
    output['seq'].append(target_seq)

    max_identity = 0
    count = 0
    max_id_seq = ''
    for seq1 in training_parse_seq:
        identity = pairwise_align_score(seq1, target_seq)
        if identity > max_identity:
            max_identity = identity
            max_id_seq = seq1
        count += 1
    
    pLDDT_score = round(pLDDT(target_seq, model), 2)
    print ("pLDDT = ", round(pLDDT_score * 100, 2))
    print ("max_identity = ", round(max_identity * 100, 2))
    output['maxID'].append(round(max_identity * 100, 2))
    output['pLDDT'].append(round(pLDDT_score * 100, 2))
    
# Convert the dictionary into DataFrame
output_df = pd.DataFrame(output)

# Print the DataFrame
output_df.head(10)

In [ ]:
# Code to get max seq identity with the training set and pLDDT for a given sequence.

model = get_esm_model()

training_parse_seq = parse_sequences('training.txt')
output = { 'maxID': [], 'index': [], 'seq': [], 'pLDDT' :[]}

# iterate over ALL of filtered (instead of a hand-picked indices list)
for idx in range(1, len(filtered) + 1):
    _idx = idx - 1  # dataframe indexes from 0.
    target_seq = filtered.iloc[_idx]['sequence']
    print('seq =', target_seq)
    print('idx = ', idx, '| len = ', len(target_seq))
    
    # compute pLDDT first so we can filter early
    pLDDT_score = round(pLDDT(target_seq, model), 2)
    print("pLDDT = ", round(pLDDT_score * 100, 2))

    # apply pLDDT >= 70 check (note: compare in percent scale)
    if (pLDDT_score * 100) < 70:
        continue

    output['index'].append(idx)
    output['seq'].append(target_seq)

    max_identity = 0
    count = 0
    max_id_seq = ''
    for seq1 in training_parse_seq:
        identity = pairwise_align_score(seq1, target_seq)
        if identity > max_identity:
            max_identity = identity
            max_id_seq = seq1
        count += 1
    
    print("max_identity = ", round(max_identity * 100, 2))
    output['maxID'].append(round(max_identity * 100, 2))
    output['pLDDT'].append(round(pLDDT_score * 100, 2))
    
# Convert the dictionary into DataFrame
output_df = pd.DataFrame(output)

# Print the DataFrame
output_df.head(10)

 # skip if too similar to training set
if (max_identity * 100) > 60:
    continue
